# Research Endpoint Stress Test

This notebook stress-tests the `/api/research/sessions` endpoint to validate the performance optimizations.

## What we're testing:

1. **Various page sizes** - Small (10) to large (100) pages
2. **Response times** - Ensure no timeouts and reasonable performance
3. **Concurrent requests** - Multiple simultaneous requests
4. **Different filters** - Test various query parameter combinations
5. **Bulk operations** - Fetching many sessions by ID
6. **Data completeness** - Verify batched data is correct

## Expected Results:

After the N+1 query fix:
- Response times should be < 5 seconds for typical requests
- No timeouts even with large page sizes
- Consistent performance across different filters
- All data fields properly populated

In [ ]:
# Setup
import sys
sys.path.insert(0, '../src')

import time
import statistics
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any
from dotenv import load_dotenv

load_dotenv()

from comparative_judging import SocialRVClient

# Initialize client
client = SocialRVClient()
print(f"✅ Client initialized, connecting to: {client.base_url}")

# Helper function to measure time
def time_request(func, *args, **kwargs):
    """Execute a function and return (result, elapsed_time)"""
    start = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - start
    return result, elapsed

# Store results for final report
test_results = []

## Test 1: Baseline - Small Page Sizes

Start with typical small requests to establish baseline performance.

In [ ]:
print("🧪 Test 1: Small page sizes (10, 25, 50)\n")

page_sizes = [10, 25, 50]
times = []

for size in page_sizes:
    result, elapsed = time_request(
        client.list_sessions,
        page=1,
        page_size=size
    )
    times.append(elapsed)
    
    print(f"📦 Page size {size:3d}: {elapsed:.2f}s | Returned {len(result['sessions'])} sessions")
    
    # Validate data completeness
    if result['sessions']:
        sample = result['sessions'][0]
        has_media = len(sample.session_media_urls) > 0
        has_target = sample.target_description is not None
        has_decoys = len(sample.decoy_ids) > 0
        print(f"   ✓ Data: Media={has_media}, Target={has_target}, Decoys={has_decoys}")

avg_time = statistics.mean(times)
print(f"\n📊 Average response time: {avg_time:.2f}s")

test_results.append({
    'test': 'Small pages',
    'avg_time': avg_time,
    'max_time': max(times),
    'passed': max(times) < 10  # Should be well under 10s
})

## Test 2: Large Page Sizes (Stress Test)

Test with large page sizes that would have caused timeouts before the fix.

In [ ]:
print("🧪 Test 2: Large page sizes (75, 100)\n")

page_sizes = [75, 100]
times = []

for size in page_sizes:
    print(f"📦 Fetching {size} sessions...", end=" ")
    
    try:
        result, elapsed = time_request(
            client.list_sessions,
            page=1,
            page_size=size
        )
        times.append(elapsed)
        
        print(f"✅ {elapsed:.2f}s | Returned {len(result['sessions'])} sessions")
        
        # Count sessions with complete data
        with_media = sum(1 for s in result['sessions'] if len(s.session_media_urls) > 0)
        with_target = sum(1 for s in result['sessions'] if s.target_description is not None)
        with_decoys = sum(1 for s in result['sessions'] if len(s.decoy_ids) > 0)
        
        print(f"   ✓ Complete data: {with_media} with media, {with_target} with targets, {with_decoys} with decoys")
        
    except Exception as e:
        print(f"❌ FAILED: {e}")
        times.append(999)  # Timeout/failure marker

if times:
    avg_time = statistics.mean([t for t in times if t < 999])
    print(f"\n📊 Average response time: {avg_time:.2f}s")
    
    test_results.append({
        'test': 'Large pages',
        'avg_time': avg_time,
        'max_time': max([t for t in times if t < 999]),
        'passed': all(t < 30 for t in times)  # Should complete within 30s (was timing out at 60s+)
    })

## Test 3: Multiple Filters (Complex Queries)

Test different filter combinations to ensure consistent performance.

In [ ]:
print("🧪 Test 3: Various filter combinations\n")

test_cases = [
    {
        'name': 'Default (public, submitted)',
        'params': {'page_size': 25}
    },
    {
        'name': 'Include non-public',
        'params': {'page_size': 25, 'include_non_public': True}
    },
    {
        'name': 'Sort by CJ rank',
        'params': {'page_size': 25, 'sort_key': 'CJ_RANK', 'sort_direction': 'asc'}
    },
    {
        'name': 'Sort by community score',
        'params': {'page_size': 25, 'sort_key': 'COMMUNITY_SCORE', 'sort_direction': 'desc'}
    },
    {
        'name': 'Include unsubmitted',
        'params': {'page_size': 25, 'include_unsubmitted': True}
    },
]

times = []

for test_case in test_cases:
    result, elapsed = time_request(
        client.list_sessions,
        **test_case['params']
    )
    times.append(elapsed)
    
    print(f"📋 {test_case['name']:30s}: {elapsed:.2f}s | {len(result['sessions'])} sessions")

avg_time = statistics.mean(times)
std_dev = statistics.stdev(times) if len(times) > 1 else 0

print(f"\n📊 Average: {avg_time:.2f}s | Std Dev: {std_dev:.2f}s")
print(f"   Consistency check: {'✅ PASS' if std_dev < 2 else '⚠️  High variance'}")

test_results.append({
    'test': 'Filter combinations',
    'avg_time': avg_time,
    'std_dev': std_dev,
    'passed': std_dev < 3  # Performance should be consistent across filters
})

## Test 4: Bulk ID Fetching

Test fetching multiple sessions by ID (up to the 100 ID limit).

In [ ]:
print("🧪 Test 4: Bulk session fetching by ID\n")

# First, get a list of session IDs
initial_result = client.list_sessions(page=1, page_size=100)
all_ids = [s.id for s in initial_result['sessions']]

print(f"📋 Collected {len(all_ids)} session IDs for bulk testing\n")

bulk_tests = [
    ('Small batch (10 IDs)', all_ids[:10]),
    ('Medium batch (50 IDs)', all_ids[:50]),
    ('Large batch (100 IDs)', all_ids[:100]),
]

times = []

for name, ids in bulk_tests:
    if len(ids) == 0:
        print(f"⏭️  {name}: Skipped (not enough IDs)")
        continue
    
    result, elapsed = time_request(
        client.get_sessions_by_ids,
        ids
    )
    times.append(elapsed)
    
    print(f"📦 {name:25s}: {elapsed:.2f}s | Found {len(result['found_ids'])}/{len(ids)} sessions")
    
    # Validate data completeness for bulk fetch
    if result['sessions']:
        with_complete_data = sum(
            1 for s in result['sessions']
            if (s.submission_time and len(s.session_media_urls) > 0 and s.target_description)
        )
        print(f"   ✓ {with_complete_data}/{len(result['sessions'])} sessions have complete data")

if times:
    avg_time = statistics.mean(times)
    print(f"\n📊 Average response time: {avg_time:.2f}s")
    
    test_results.append({
        'test': 'Bulk ID fetch',
        'avg_time': avg_time,
        'max_time': max(times),
        'passed': max(times) < 15  # Should handle 100 IDs efficiently
    })

## Test 5: Concurrent Requests (Load Test)

Simulate multiple researchers hitting the endpoint simultaneously.

In [ ]:
print("🧪 Test 5: Concurrent requests\n")

def fetch_page(page_num):
    """Fetch a single page and return (page_num, elapsed_time, success)"""
    try:
        start = time.time()
        result = client.list_sessions(page=page_num, page_size=25)
        elapsed = time.time() - start
        return (page_num, elapsed, True, len(result['sessions']))
    except Exception as e:
        return (page_num, 0, False, str(e))

# Test with 5 concurrent requests
num_concurrent = 5
print(f"🚀 Launching {num_concurrent} concurrent requests...\n")

start_time = time.time()

with ThreadPoolExecutor(max_workers=num_concurrent) as executor:
    futures = [executor.submit(fetch_page, i+1) for i in range(num_concurrent)]
    
    results = []
    for future in as_completed(futures):
        results.append(future.result())

total_time = time.time() - start_time

# Sort by page number for display
results.sort(key=lambda x: x[0])

successful = [r for r in results if r[2]]
failed = [r for r in results if not r[2]]

print("📊 Results:")
for page_num, elapsed, success, data in results:
    if success:
        print(f"   Page {page_num}: ✅ {elapsed:.2f}s | {data} sessions")
    else:
        print(f"   Page {page_num}: ❌ Failed - {data}")

print(f"\n⏱️  Total wall time: {total_time:.2f}s")
print(f"✅ Success rate: {len(successful)}/{num_concurrent} ({len(successful)/num_concurrent*100:.0f}%)")

if successful:
    avg_request_time = statistics.mean([r[1] for r in successful])
    print(f"📊 Average request time: {avg_request_time:.2f}s")
    
    test_results.append({
        'test': 'Concurrent requests',
        'total_time': total_time,
        'avg_request_time': avg_request_time,
        'success_rate': len(successful)/num_concurrent,
        'passed': len(successful) == num_concurrent and avg_request_time < 10
    })

## Test 6: Pagination Stress Test

Rapidly fetch multiple pages to test sustained performance.

In [ ]:
print("🧪 Test 6: Rapid pagination\n")

num_pages = 10
page_size = 25

print(f"📄 Fetching {num_pages} pages of {page_size} sessions each...\n")

times = []
total_sessions = 0

for page in range(1, num_pages + 1):
    result, elapsed = time_request(
        client.list_sessions,
        page=page,
        page_size=page_size
    )
    times.append(elapsed)
    total_sessions += len(result['sessions'])
    
    print(f"   Page {page:2d}: {elapsed:.2f}s")

total_time = sum(times)
avg_time = statistics.mean(times)
std_dev = statistics.stdev(times) if len(times) > 1 else 0

print(f"\n📊 Statistics:")
print(f"   Total time: {total_time:.2f}s")
print(f"   Average per page: {avg_time:.2f}s")
print(f"   Std deviation: {std_dev:.2f}s")
print(f"   Total sessions fetched: {total_sessions}")
print(f"   Throughput: {total_sessions/total_time:.1f} sessions/sec")

test_results.append({
    'test': 'Rapid pagination',
    'avg_time': avg_time,
    'std_dev': std_dev,
    'throughput': total_sessions/total_time,
    'passed': avg_time < 5 and std_dev < 2  # Consistent performance
})

## Test 7: Data Integrity Validation

Verify that the batch optimization didn't break data completeness.

In [ ]:
print("🧪 Test 7: Data integrity validation\n")

# Fetch a good sample size
result, elapsed = time_request(
    client.list_sessions,
    page=1,
    page_size=50
)

sessions = result['sessions']
print(f"📊 Analyzing {len(sessions)} sessions...\n")

# Count various data completeness metrics
submitted = [s for s in sessions if s.submission_time]
with_media = [s for s in submitted if len(s.session_media_urls) > 0]
with_target = [s for s in submitted if s.target_description]
with_target_image = [s for s in submitted if s.target_image_url]
with_decoys = [s for s in submitted if len(s.decoy_ids) > 0]
with_all_urls = [s for s in with_media if all(m.get('url') for m in s.session_media_urls)]

print("✅ Data Completeness:")

if len(sessions) == 0:
    print("   ⚠️  No sessions found! Check filters or run diagnostic notebook (00_diagnose_endpoint.ipynb)")
else:
    print(f"   Submitted sessions: {len(submitted)}/{len(sessions)} ({len(submitted)/len(sessions)*100:.0f}%)")

if submitted:
    print(f"   Sessions with media files: {len(with_media)}/{len(submitted)} ({len(with_media)/len(submitted)*100:.0f}%)")
    print(f"   Sessions with target data: {len(with_target)}/{len(submitted)} ({len(with_target)/len(submitted)*100:.0f}%)")
    print(f"   Sessions with target images: {len(with_target_image)}/{len(submitted)} ({len(with_target_image)/len(submitted)*100:.0f}%)")
    print(f"   Sessions with decoys: {len(with_decoys)}/{len(submitted)} ({len(with_decoys)/len(submitted)*100:.0f}%)")
    
    if with_media:
        print(f"   Media with valid URLs: {len(with_all_urls)}/{len(with_media)} ({len(with_all_urls)/len(with_media)*100:.0f}%)")
        
        # Check media URL format
        sample_media = with_media[0].session_media_urls[0]
        has_url = 'url' in sample_media and sample_media['url']
        has_mime = 'mime_type' in sample_media
        has_path = 'storage_path' in sample_media
        
        print(f"\n✅ Media structure validation:")
        print(f"   Has 'url' field: {has_url}")
        print(f"   Has 'mime_type' field: {has_mime}")
        print(f"   Has 'storage_path' field: {has_path}")

# Check for proper data types
print(f"\n✅ Data type validation:")

if len(sessions) == 0:
    print("   ⚠️  Cannot validate data types - no sessions found")
    all_checks_passed = False
else:
    sample = sessions[0]
    checks = [
        ('id is string', isinstance(sample.id, str)),
        ('user_id is string', isinstance(sample.user_id, str)),
        ('session_media_urls is list', isinstance(sample.session_media_urls, list)),
        ('decoy_ids is list', isinstance(sample.decoy_ids, list)),
        ('num_comments is int', isinstance(sample.num_comments, int)),
    ]
    
    for check_name, passed in checks:
        status = "✅" if passed else "❌"
        print(f"   {status} {check_name}")
    
    all_checks_passed = all(check[1] for check in checks)

test_results.append({
    'test': 'Data integrity',
    'submitted_pct': len(submitted)/len(sessions) if sessions else 0,
    'media_pct': len(with_media)/len(submitted) if submitted else 0,
    'target_pct': len(with_target)/len(submitted) if submitted else 0,
    'passed': all_checks_passed and (len(with_media)/len(submitted) > 0.9 if submitted else True)
})

## Test 8: Full Database Pagination with Failure Tracking (PARALLELIZED)

Paginate through the ENTIRE database at maximum page size (100) using **parallel requests** for maximum speed.

### What we track:
- How many requests fail
- Which pages/sessions failed to load
- Total session count and unique session IDs
- Data completeness (missing media, targets, decoys)
- Performance metrics and speedup from parallelization

### Parallelization:
- Uses ThreadPoolExecutor with configurable concurrent requests
- Thread-safe tracking of all metrics
- Progress updates every 10 pages
- Measures speedup vs sequential fetching

In [ ]:
print("🧪 Test 8: Full database pagination at max page size (PARALLELIZED)\n")

MAX_PAGE_SIZE = 100
TIMEOUT_SECONDS = 60  # Consider a request failed if it takes longer than this
MAX_CONCURRENT_REQUESTS = 10  # Number of parallel requests

# Track all failures and successes (using thread-safe operations)
from threading import Lock
results_lock = Lock()

failed_pages = []
failed_requests = 0
successful_requests = 0
total_sessions_loaded = 0
all_session_ids = set()
page_times = []

# Track sessions with incomplete data
sessions_missing_media = []
sessions_missing_target = []
sessions_missing_decoys = []

print(f"📦 Fetching entire database with page_size={MAX_PAGE_SIZE}")
print(f"⏱️  Timeout threshold: {TIMEOUT_SECONDS}s\n")

# First, get the total count
try:
    first_page_result, first_elapsed = time_request(
        client.list_sessions,
        page=1,
        page_size=MAX_PAGE_SIZE,
        include_non_public=True,  # Get everything
        include_unsubmitted=True
    )
    
    total_count = first_page_result['total_count']
    total_pages = first_page_result['total_pages']
    
    print(f"📊 Database stats:")
    print(f"   Total sessions: {total_count}")
    print(f"   Total pages: {total_pages}")
    print(f"   Page size: {MAX_PAGE_SIZE}\n")
    
    # Process first page
    page_times.append(first_elapsed)
    successful_requests += 1
    total_sessions_loaded += len(first_page_result['sessions'])
    
    # Track session IDs
    for session in first_page_result['sessions']:
        all_session_ids.add(session.id)
        
        # Check for incomplete data
        if not session.session_media_urls or len(session.session_media_urls) == 0:
            sessions_missing_media.append((1, session.id))
        if not session.target_description:
            sessions_missing_target.append((1, session.id))
        if not session.decoy_ids or len(session.decoy_ids) == 0:
            sessions_missing_decoys.append((1, session.id))
    
    print(f"✅ Page 1/{total_pages}: {first_elapsed:.2f}s | {len(first_page_result['sessions'])} sessions")
    
    # Now fetch remaining pages
    for page in range(2, total_pages + 1):
        try:
            result, elapsed = time_request(
                client.list_sessions,
                page=page,
                page_size=MAX_PAGE_SIZE,
                include_non_public=True,
                include_unsubmitted=True
            )
            
            page_times.append(elapsed)
            
            # Check if request took too long (potential timeout)
            if elapsed > TIMEOUT_SECONDS:
                print(f"⚠️  Page {page}/{total_pages}: {elapsed:.2f}s (SLOW!) | {len(result['sessions'])} sessions")
                failed_pages.append({
                    'page': page,
                    'reason': 'timeout',
                    'elapsed': elapsed
                })
                failed_requests += 1
            else:
                print(f"✅ Page {page}/{total_pages}: {elapsed:.2f}s | {len(result['sessions'])} sessions")
                successful_requests += 1
            
            total_sessions_loaded += len(result['sessions'])
            
            # Track session IDs and check for incomplete data
            for session in result['sessions']:
                all_session_ids.add(session.id)
                
                if not session.session_media_urls or len(session.session_media_urls) == 0:
                    sessions_missing_media.append((page, session.id))
                if not session.target_description:
                    sessions_missing_target.append((page, session.id))
                if not session.decoy_ids or len(session.decoy_ids) == 0:
                    sessions_missing_decoys.append((page, session.id))
            
        except Exception as e:
            print(f"❌ Page {page}/{total_pages}: FAILED - {str(e)[:100]}")
            failed_pages.append({
                'page': page,
                'reason': 'exception',
                'error': str(e)
            })
            failed_requests += 1
        
        # Small delay to avoid overwhelming the server
        time.sleep(0.1)
    
except Exception as e:
    print(f"❌ CRITICAL ERROR: Could not fetch first page: {e}")
    total_count = 0
    total_pages = 0

print(f"\n{'='*70}")
print("📊 FULL DATABASE PAGINATION RESULTS")
print(f"{'='*70}\n")

if total_count > 0:
    print(f"✅ Successful requests: {successful_requests}/{successful_requests + failed_requests} ({successful_requests/(successful_requests + failed_requests)*100:.1f}%)")
    print(f"❌ Failed requests: {failed_requests}")
    print(f"📦 Total sessions loaded: {total_sessions_loaded}/{total_count} ({total_sessions_loaded/total_count*100:.1f}%)")
    print(f"🔢 Unique session IDs: {len(all_session_ids)}")
    
    if page_times:
        print(f"\n⏱️  Performance:")
        print(f"   Average time per page: {statistics.mean(page_times):.2f}s")
        print(f"   Fastest page: {min(page_times):.2f}s")
        print(f"   Slowest page: {max(page_times):.2f}s")
        print(f"   Std deviation: {statistics.stdev(page_times) if len(page_times) > 1 else 0:.2f}s")
        print(f"   Total time: {sum(page_times):.2f}s")
    
    # Report missing sessions
    missing_sessions = total_count - total_sessions_loaded
    if missing_sessions > 0:
        print(f"\n⚠️  WARNING: {missing_sessions} sessions were NOT loaded!")
        print(f"   This could indicate:")
        print(f"   - Failed requests that didn't return sessions")
        print(f"   - Database inconsistencies")
        print(f"   - Pagination issues")
    
    # Report failed pages
    if failed_pages:
        print(f"\n❌ Failed pages detail:")
        for failure in failed_pages:
            page_num = failure['page']
            reason = failure['reason']
            if reason == 'timeout':
                print(f"   Page {page_num}: Timeout/Slow ({failure['elapsed']:.2f}s)")
            else:
                error_msg = failure.get('error', 'Unknown error')[:80]
                print(f"   Page {page_num}: {error_msg}")
    
    # Report incomplete data
    print(f"\n📋 Data completeness:")
    print(f"   Sessions missing media: {len(sessions_missing_media)}")
    print(f"   Sessions missing target: {len(sessions_missing_target)}")
    print(f"   Sessions missing decoys: {len(sessions_missing_decoys)}")
    
    if sessions_missing_media and len(sessions_missing_media) <= 10:
        print(f"\n   Sessions missing media (page, id):")
        for page, session_id in sessions_missing_media[:10]:
            print(f"      Page {page}: {session_id}")
    
    if sessions_missing_target and len(sessions_missing_target) <= 10:
        print(f"\n   Sessions missing target (page, id):")
        for page, session_id in sessions_missing_target[:10]:
            print(f"      Page {page}: {session_id}")
    
    test_passed = (
        failed_requests == 0 and 
        missing_sessions == 0 and
        len(all_session_ids) == total_count
    )
    
    test_results.append({
        'test': 'Full DB pagination',
        'total_sessions': total_count,
        'loaded_sessions': total_sessions_loaded,
        'unique_ids': len(all_session_ids),
        'failed_requests': failed_requests,
        'success_rate': successful_requests/(successful_requests + failed_requests) if (successful_requests + failed_requests) > 0 else 0,
        'avg_time': statistics.mean(page_times) if page_times else 0,
        'passed': test_passed
    })
    
    if test_passed:
        print(f"\n✅ FULL DATABASE SCAN SUCCESSFUL!")
    else:
        print(f"\n⚠️  FULL DATABASE SCAN HAD ISSUES - See details above")
else:
    print("❌ Could not complete full database pagination test")
    test_results.append({
        'test': 'Full DB pagination',
        'passed': False
    })

print(f"\n{'='*70}")

## Final Report: Test Results Summary

Overall assessment of the endpoint performance after optimization.

print("="*70)
print("📋 STRESS TEST SUMMARY")
print("="*70)
print()

all_passed = True

for i, result in enumerate(test_results, 1):
    status = "✅ PASS" if result['passed'] else "❌ FAIL"
    print(f"{i}. {result['test']:25s} {status}")
    
    # Print relevant metrics
    if 'avg_time' in result:
        print(f"   Avg time: {result['avg_time']:.2f}s", end="")
    if 'max_time' in result:
        print(f" | Max: {result['max_time']:.2f}s", end="")
    if 'std_dev' in result:
        print(f" | StdDev: {result['std_dev']:.2f}s", end="")
    if 'throughput' in result:
        print(f" | Throughput: {result['throughput']:.1f} sessions/s", end="")
    if 'success_rate' in result:
        print(f" | Success: {result['success_rate']*100:.0f}%", end="")
    if 'total_sessions' in result:
        print(f" | Total: {result['total_sessions']}", end="")
    if 'loaded_sessions' in result:
        print(f" | Loaded: {result['loaded_sessions']}", end="")
    if 'failed_requests' in result and result['failed_requests'] > 0:
        print(f" | Failed: {result['failed_requests']}", end="")
    print()
    
    if not result['passed']:
        all_passed = False
    print()

print("="*70)

if all_passed:
    print("\n🎉 ALL TESTS PASSED! 🎉")
    print("\nThe endpoint optimizations are working as expected:")
    print("  ✅ No timeouts detected")
    print("  ✅ Response times are fast and consistent")
    print("  ✅ Data integrity maintained")
    print("  ✅ Handles concurrent load well")
    print("  ✅ Batch optimization eliminated N+1 query problem")
    print("  ✅ Successfully paginated through entire database")
else:
    print("\n⚠️  SOME TESTS FAILED")
    print("\nFailed tests indicate potential issues that need investigation.")
    print("Review the detailed output above for specific failure points.")

print("\n" + "="*70)

## Bonus: Look Up a Specific Session by ID

Demonstrates how to fetch a single session when you know its session_id.

In [ ]:
print("🔍 Looking up sessions by ID - Comparing Two Methods\n")

# Get some example session IDs
example_ids = []
if 'all_session_ids' in globals() and all_session_ids:
    # Use session IDs from our full database scan
    example_ids = list(all_session_ids)[:3]  # Get 3 IDs for testing
    print(f"📌 Using {len(example_ids)} session IDs from earlier test")
else:
    # Fetch some sessions to get IDs
    result = client.list_sessions(page=1, page_size=3)
    example_ids = [s.id for s in result['sessions']]
    print(f"📌 Fetched {len(example_ids)} sample session IDs")

if not example_ids:
    print("⚠️  No sessions available to demonstrate lookup")
else:
    print(f"\n{'='*70}")
    print("METHOD 1: get_session() - Single ID lookup using 'id' parameter")
    print(f"{'='*70}\n")
    
    single_id = example_ids[0]
    print(f"🔎 Fetching session {single_id} using get_session()...")
    
    try:
        start = time.time()
        session = client.get_session(single_id)
        elapsed = time.time() - start
        
        print(f"✅ Success! Fetched in {elapsed:.2f}s\n")
        
        print("📊 Session Details:")
        print(f"   ID: {session.id}")
        print(f"   User: {session.user_display_name}")
        print(f"   Target: {session.target_description[:100] + '...' if session.target_description and len(session.target_description) > 100 else session.target_description or 'N/A'}")
        print(f"   Coordinate: {session.target_coordinate}")
        print(f"   Submitted: {session.submission_time if session.submission_time else 'Not submitted'}")
        print(f"   Media files: {len(session.session_media_urls)}")
        print(f"   Decoys: {len(session.decoy_ids)}")
        
        if session.comparative_judging_rank:
            print(f"   CJ Rank: {session.comparative_judging_rank}")
        
    except Exception as e:
        print(f"❌ FAILED: {str(e)}")
        print(f"   Error type: {type(e).__name__}")
        print(f"   This is the error you saw in the logs (500 with 'id' parameter)")
    
    print(f"\n{'='*70}")
    print("METHOD 2: get_sessions_by_ids() - Bulk lookup using 'ids' parameter")
    print(f"{'='*70}\n")
    
    print(f"🔎 Fetching {len(example_ids)} sessions using get_sessions_by_ids()...")
    print(f"   IDs: {[id[:8] + '...' for id in example_ids]}\n")
    
    try:
        start = time.time()
        result = client.get_sessions_by_ids(example_ids)
        elapsed = time.time() - start
        
        print(f"✅ Success! Fetched in {elapsed:.2f}s\n")
        
        print("📊 Results:")
        print(f"   Requested: {len(example_ids)} sessions")
        print(f"   Found: {len(result['found_ids'])} sessions")
        print(f"   Missing: {len(result['missing_ids'])} sessions")
        
        if result['missing_ids']:
            print(f"   Missing IDs: {result['missing_ids']}")
        
        print(f"\n📋 Session Details:")
        for i, session in enumerate(result['sessions'][:3], 1):  # Show first 3
            print(f"\n   {i}. {session.id}")
            print(f"      User: {session.user_display_name}")
            print(f"      Target: {session.target_description[:80] + '...' if session.target_description and len(session.target_description) > 80 else session.target_description or 'N/A'}")
            print(f"      CJ Rank: {session.comparative_judging_rank if session.comparative_judging_rank else 'N/A'}")
            print(f"      Media files: {len(session.session_media_urls)}")
        
        # Show media URLs for first session
        if result['sessions'] and result['sessions'][0].session_media_urls:
            print(f"\n📎 Media files (first session):")
            for i, media in enumerate(result['sessions'][0].session_media_urls, 1):
                mime_type = media.get('mime_type', 'unknown')
                url = media.get('url', 'N/A')
                print(f"      {i}. {mime_type}: {url[:70]}...")
        
    except Exception as e:
        print(f"❌ FAILED: {str(e)}")
        print(f"   Error type: {type(e).__name__}")

print("\n" + "="*70)
print("💡 RECOMMENDATION:")
print("   Use get_sessions_by_ids() (plural) for reliable fetching:")
print("   - Works for 1 or more sessions (up to 100)")
print("   - Returns found_ids and missing_ids for error handling")
print("   - More resilient than get_session() (singular)")
print("")
print("   Example: client.get_sessions_by_ids([session_id_1, session_id_2, ...])")
print("="*70)